In [2]:
import os
from  pathlib import Path
import sys
import pandas as pd
from sqlalchemy import text
from gspread.utils import rowcol_to_a1
from gspread_dataframe import set_with_dataframe

# Установка базовой директории и пути к файлу с учетными данными. Используем конструкцию try-except для обработки возможных ошибок при определении пути для notebook.
try:
    BASE_DIR = Path(__file__).resolve().parents[2]
except NameError:
    BASE_DIR = Path.cwd().resolve().parents[1] 

SRC_DIR = os.path.join(BASE_DIR, 'src')

# Добавляем src в пути поиска модулей
if SRC_DIR not in sys.path:
    sys.path.append(SRC_DIR)

from core.google_sheets_scheme import week_n_redeem
from core.utils_gspread import safe_open_spreadsheet
from core.utils_sql import get_db_engine

In [ ]:
def update_week_n_redeem():
	# Получаем подключение к базе данных
	engine = get_db_engine()
	# Выполняем SQL-запрос для получения данных по еженедельным отчетам реализации и уведомлениях о выпкупе
	query = text("""WITH week_rep AS ( -- Данные еженедельного отчета реализации
	SELECT 
		-- Достаю данные по различным статьям еженедельного отчета в разрезе документа
		SUM(
		CASE 
			WHEN w.title = 'Итого стоимость реализованного товара и услуг'
				THEN COALESCE(sum_rub, 0)
		END) AS total_sum,
			ROUND(SUM(
		CASE 
			WHEN w.title = 'Итого стоимость реализованного товара и услуг'
				THEN COALESCE(sum_rub, 0)
		END)*100/107, 2) AS total_sum_without_vat,
			SUM(
		CASE 
			WHEN w.title IN ('Компенсация ущерба', 'Прочие выплаты')
				THEN COALESCE(sum_rub, 0)
		END) AS damages_comp_other,
			SUM(
		CASE 
			WHEN w.title = 'Компенсация скидки по программе лояльности'
				THEN COALESCE(sum_rub, 0)
		END) AS discount_loyalty,
		SUM(
		CASE 
			WHEN w.title IN ('Сумма вознаграждения Вайлдберриз за текущий период (ВВ), без НДС', 'НДС с вознаграждения Вайлдберриз')
				THEN COALESCE(sum_rub, 0)
		END) AS award,
		SUM(
		CASE 
			WHEN w.title IN ('Сумма, удержанная в счёт обеспечения организации платежа')
				THEN COALESCE(sum_rub, 0)
		END) AS amount_withheld_to_org,
		SUM(
		CASE 
			WHEN w.title IN ('Возмещение расходов по перевозке')
				THEN COALESCE(sum_rub, 0)
		END) AS reimbursement_of_transp_costs,
		SUM(
		CASE 
			WHEN w.title IN ('Возмещение за выдачу и возврат товаров на ПВЗ')
				THEN COALESCE(sum_rub, 0)
		END) AS reimbursement_for_delivery_and_return_of_goods_to_pvz,	
		SUM(
		CASE 
			WHEN w.title IN ('Штрафы')
				THEN COALESCE(sum_rub, 0)
		END) AS penalties,
		SUM(
		CASE 
			WHEN w.title IN ('Прочие удержания')
				THEN COALESCE(sum_rub, 0)
		END) other_deductions,	
		SUM(
		CASE 
			WHEN w.title IN ('Удержания в пользу третьих лиц')
				THEN COALESCE(sum_rub, 0)
		END) retentions_in_favor_of_third_parties,	
		-- Выделяю данные для группировки
		w.doc_num AS weekly_rep,
		DATE(DATE_TRUNC('week', w."date")) AS week_start,
		DATE((DATE_TRUNC('week', w."date") + INTERVAL '7 days' - INTERVAL '1 microsecond')) AS week_end,
		w.account
	FROM weekly_implementation_report w
	GROUP BY w.doc_num, w."date", w.account),
	fin_rep AS -- Данные еженедельного финансового отчета
		(SELECT 
			f.realizationreport_id,
			f.date_from,
			f.date_to,
			sum
			(CASE
				WHEN f.doc_type_name ILIKE '%возврат%'
				THEN (f.ppvz_for_pay)
				ELSE 0
			END) AS return_pay,
			f.account
		FROM fin_reports_full f
		WHERE f.report_type = 2
		GROUP BY f.realizationreport_id, f.date_from, f.date_to, f.account),
	redeem_not AS( -- Данные уведомления о выкупе
		SELECT sum(sum_rub_with_vat) AS sum_rub_with_vat,
		ROUND(sum(sum_rub_with_vat)*100/107,2) AS sum_rub_without_vat,
		SUBSTRING(r.doc_name FROM '№(\d+)') AS redeem_notif -- из полного названия документа извлекаю номер
	FROM redeem_notification r
	GROUP BY r.doc_name
	)
	SELECT 
		w.account,
		w.week_start AS "Начало_недели",
		w.week_end AS "Конец_недели", 
		w.weekly_rep AS "Номер_еженедельного_отчета",
		w.total_sum AS "Всего_товара",
		w.total_sum_without_vat AS "Всего_товара_БЕЗ_НДС",
		w.damages_comp_other AS "Компенсации_ущерба_и_прочие_выплаты",
		w.discount_loyalty AS "Компенсация_скидки_по_программе_лояльности",
		r.redeem_notif AS "Уведомление_о_выкупе_№",
		r.sum_rub_with_vat AS "Выкуплено_по_уведомлению",
		r.sum_rub_without_vat AS "Выкуплено_по_уведомлению_без_НДС",
		f.return_pay AS "Вовзрат_выкупа",
		CASE 
			WHEN w.award < 0
			THEN w.award*-1
			ELSE w.award 
		END AS "Вознагрождение_в_доход",
		CASE 
			WHEN w.award < 0
			THEN ROUND((w.award*-1)*100/107,2)
			ELSE w.award 
		END AS "Вознагрождение_в_доход_БЕЗ_НДС",	
		w.award AS "Вознаграждение",
		amount_withheld_to_org AS "Сумма_удержанная_в_счёт_обеспечения_организации_платежа",
		reimbursement_of_transp_costs AS "Возмещение расходов по перевозке",
		reimbursement_for_delivery_and_return_of_goods_to_pvz AS "Возмещение_за_выдачу_и_возврат_товаров_на_ПВЗ", 
		penalties AS "Штрафы",
		other_deductions AS "Прочие удержания",
		retentions_in_favor_of_third_parties AS "Удержания_в_пользу_третьих_лиц",
		(f.return_pay+amount_withheld_to_org+reimbursement_of_transp_costs+
		reimbursement_for_delivery_and_return_of_goods_to_pvz+penalties+
		other_deductions+retentions_in_favor_of_third_parties) AS "Итого_расходы",
		(w.total_sum+damages_comp_other+r.sum_rub_with_vat+(CASE 
			WHEN w.award < 0
			THEN w.award*-1
			ELSE w.award 
		END))- -- Доходы минус расходы
		(f.return_pay+amount_withheld_to_org+reimbursement_of_transp_costs+
		reimbursement_for_delivery_and_return_of_goods_to_pvz+penalties+
		other_deductions+retentions_in_favor_of_third_parties) AS "К_перечислению_по_отчетам"
	FROM week_rep w
	LEFT JOIN fin_rep f ON UPPER(w.account) = UPPER(f.account) 
		AND w.week_start = f.date_from
	LEFT JOIN redeem_not r
		ON f.realizationreport_id = r.redeem_notif::INT
	ORDER BY w.account, w.week_start DESC;""")

	# Загружаем данные в DataFrame
	df = pd.read_sql(query, engine)
	df = df.rename(columns={
							"Всего_товара":             "Всего_стоимость_реализованного_товара",
							"Всего_товара_БЕЗ_НДС": "Всего_стоимость_реализованного_товара_без_НДС",
							"Компенсации_ущерба_и_прочие_выпла": "Компенсации_ущерба_и_прочие_выплаты",
							"Компенсация_скидки_по_программе_л": "Компенсация_скидки_по_программе_лояльности",
							"Возмещение_за_выдачу_и_возврат_тов": "Возмещение_за_выдачу_и_возврат_товара_ПВЗ",
							"Сумма_удержанная_в_счёт_обеспечен":"Сумма_удержанная_в_счёт_обеспечения_организации_платежа",
							"Возмещение_за_выдачу_и_возврат_тов":"Возмещение_за_выдачу_и_возврат_товаров_на_ПВЗ"
							})
	# Сортировка по неделям и аккаунтам
	df = df.sort_values(by=['Начало_недели', 'account'], ascending=[False, True])
	return df
	# # Открываем таблицу 
	# report_table = safe_open_spreadsheet(week_n_redeem['title'])
	# # Обновляем данные гугл таблицы
	# report_sheet = report_table.worksheet(week_n_redeem['data'])
	# set_with_dataframe(report_sheet, df)

<>:88: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
<>:88: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
C:\Users\123\AppData\Local\Temp\ipykernel_30680\3854951588.py:88: SyntaxWarning: "\d" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\d"? A raw string is also an option.
  SUBSTRING(r.doc_name FROM '№(\d+)') AS redeem_notif -- из полного названия документа извлекаю номер


In [25]:
df = update_week_n_redeem()

In [31]:
df = df.sort_values(by=['Начало_недели', 'account'], ascending=[True, True])
df.head(10)

,account,Начало_недели,Конец_недели,Номер_еженедельного_отчета,Всего_стоимость_реализованного_товара,Всего_стоимость_реализованного_товара_без_НДС,Компенсации_ущерба_и_прочие_выплаты,Компенсация_скидки_по_программе_лояльности,Уведомление_о_выкупе_№,Выкуплено_по_уведомлению,...,Вознагрождение_в_доход_БЕЗ_НДС,Вознаграждение,Сумма_удержанная_в_счёт_обеспечения_организации_платежа,Возмещение расходов по перевозке,Возмещение_за_выдачу_и_возврат_товаров_на_ПВЗ,Штрафы,Прочие удержания,Удержания_в_пользу_третьих_лиц,Итого_расходы,К_перечислению_по_отчетам
7,ВЕКТОР,2025-12-29,2026-01-04,588492200.0,8206573.02,7669694.41,23549.81,2309.77,588492204,524313.33,...,750343.00,-802867.01,NaN,326985.52,195907.17,7492.71,144.0,1281489.08,NaN,NaN
6,ВЕКТОР,2025-12-29,2026-01-04,588120847.0,3888671.82,3634272.73,0.00,1640.73,588492204,524313.33,...,627206.10,-671110.53,88543.52,234615.52,94750.86,34955.15,1909736.0,0.00,2363657.89,2720437.79
14,ВЕКТОР2,2025-12-29,2026-01-04,587529407.0,1428792.33,1335319.93,0.00,165.83,588601582,136867.87,...,236676.11,-253243.44,31183.92,87775.97,33433.53,469181.71,811800.6,0.00,1434897.76,384005.88
15,ВЕКТОР2,2025-12-29,2026-01-04,588601572.0,2866593.97,2679059.79,2566.49,602.66,588601582,136867.87,...,341143.40,-365023.44,NaN,119682.57,68057.25,66468.10,547.2,0.00,NaN,NaN
37,ДАНИЕЛЯН,2025-12-29,2026-01-04,588309905.0,2140071.18,2000066.52,7617.90,146.85,588309907,128625.20,...,237753.93,-254396.71,NaN,110094.94,50986.10,230.55,18990.0,458016.91,NaN,NaN
36,ДАНИЕЛЯН,2025-12-29,2026-01-04,587997691.0,1081393.00,1010647.66,0.00,467.86,588309907,128625.20,...,185577.36,-198567.77,24908.08,80479.37,26793.99,33987.91,415939.0,0.00,582108.35,826477.62
45,ЛОПАТИНА,2025-12-29,2026-01-04,588311512.0,3344535.89,3125734.48,864.95,737.71,588311513,280984.62,...,391445.26,-418846.43,NaN,127310.72,78087.61,2194.58,10.0,40394.00,NaN,NaN
44,ЛОПАТИНА,2025-12-29,2026-01-04,587991137.0,866103.50,809442.52,0.00,75.61,588311513,280984.62,...,142849.68,-152849.16,18691.47,64817.16,20359.87,43281.40,164441.0,0.00,313014.92,986922.36
53,ОГАНЕСЯН,2025-12-29,2026-01-04,587989294.0,1583402.35,1479815.28,0.00,2114.58,588279854,519805.74,...,290319.79,-310642.17,36216.95,116505.31,38799.18,69783.34,300911.4,0.00,566300.02,1847550.24
52,ОГАНЕСЯН,2025-12-29,2026-01-04,588279851.0,6368514.42,5951882.64,10874.80,1669.95,588279854,519805.74,...,702451.57,-751623.18,NaN,234376.73,152736.80,23185.53,704.0,491015.28,NaN,NaN


In [16]:
weekl_n_redeem()